In [1]:
import time
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F 
from pyspark.sql import types as T
from kafka import KafkaProducer

spark = SparkSession \
    .builder \
    .master("local") \
    .config("spark.driver.memory", "4g") \
    .appName("ex5_Producing to Kafka") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 23:47:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/13 23:47:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/13 23:47:55 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
reviews_df = spark.read.parquet('s3a://pyspark/data/source/google_reviews/', header=True)

26/08/13 23:47:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [3]:
reviews_df.printSchema()

root
 |-- application_name: string (nullable = true)
 |-- translated_review: string (nullable = true)
 |-- sentiment_rank: long (nullable = true)
 |-- sentiment_polarity: float (nullable = true)
 |-- sentiment_subjectivity: float (nullable = true)



In [4]:
#Convert the DataFrame records to JSON format
data = reviews_df.toJSON()
print(data.take(6))

['{"application_name":"10 Best Foods for You","translated_review":"\\"I like eat delicious food. That\'s I\'m cooking food myself, case \\"\\"10 Best Foods\\"\\" helps lot","sentiment_subjectivity":1.0}', '{"application_name":"10 Best Foods for You","translated_review":"This help eating healthy exercise regular basis","sentiment_rank":1,"sentiment_polarity":0.25,"sentiment_subjectivity":0.28846154}', '{"application_name":"10 Best Foods for You","translated_review":"nan","sentiment_polarity":"NaN","sentiment_subjectivity":"NaN"}', '{"application_name":"10 Best Foods for You","translated_review":"Works great especially going grocery store","sentiment_rank":1,"sentiment_polarity":0.4,"sentiment_subjectivity":0.875}', '{"application_name":"10 Best Foods for You","translated_review":"Best idea us","sentiment_rank":1,"sentiment_polarity":1.0,"sentiment_subjectivity":0.3}', '{"application_name":"10 Best Foods for You","translated_review":"Best way","sentiment_rank":1,"sentiment_polarity":1.0,

In [39]:
#Set up a Kafka producer
producer = KafkaProducer(bootstrap_servers='course-kafka:9092', value_serializer=lambda v: v.encode('utf-8'))

In [40]:
i = 0
for json_data in data.collect():
    i = i + 1
    producer.send(topic='gps-user-review-source', value=json_data)
    if i == 50:
        producer.flush()
        time.sleep(5)
        i = 0

In [35]:
producer.close()
spark.stop()